In [0]:
# === J5 - DistriNet feature engineering setup ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.ingestion import ingest_parquet_to_bronze_memory
from src.cleaning import clean_bronze_to_silver
from src.transformation import transform_silver_to_gold
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when

spark = SparkSession.builder.appName("j5-disternet-fe").getOrCreate()
spark.sql("USE CATALOG workspace")

print(f"✓ Spark version: {spark.version}")
print(f"✓ Modules imported")

In [0]:
# === Read the real DistriNet Monday file from the Unity Catalog Volume ===
disternet_path = "/Volumes/workspace/cybersecurity/raw_data/Benign-Monday.parquet"

print(f"Reading: {disternet_path}")

# Bronze ingestion (in memory, from Parquet)
df_bronze = ingest_parquet_to_bronze_memory(spark, disternet_path)

print(f"\n✓ DistriNet Monday loaded")
print(f"  Rows: {df_bronze.count()}")
print(f"  Columns: {len(df_bronze.columns)}")

In [0]:
# === Inspect the REAL DistriNet columns ===

print("--- All columns ---")
for i, col_name in enumerate(df_bronze.columns):
    print(f"  {i+1:3d}. {col_name}")

print(f"\n--- Schema (all columns) ---")
df_bronze.printSchema()

print(f"\n--- Column count by type ---")
type_counts = {}
for field in df_bronze.schema.fields:
    t = str(field.dataType)
    type_counts[t] = type_counts.get(t, 0) + 1
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")

print(f"\n--- Preview (first 3 rows, first 10 columns) ---")
df_bronze.select(df_bronze.columns[:10]).show(3, truncate=False)

print(f"\n--- Label distribution ---")
# Find the label column (might be 'label', 'Label', or something else)
label_candidates = [c for c in df_bronze.columns if 'label' in c.lower()]
print(f"Label candidates found: {label_candidates}")

if label_candidates:
    label_col = label_candidates[0]
    df_bronze.groupBy(label_col).count().orderBy(F.col("count").desc()).show(20, truncate=False)

In [0]:
# === Run Bronze -> Silver cleaning on REAL DistriNet data ===
print("Starting cleaning pipeline on real DistriNet data...")
print("=" * 60)

df_silver = clean_bronze_to_silver(df_bronze)

print("=" * 60)
print(f"\n✓ Silver DataFrame ready")
print(f"  Rows: {df_silver.count()}")
print(f"  Columns: {len(df_silver.columns)}")

In [0]:
# === Inspect the Silver DataFrame ===
print("--- Silver columns ---")
print(df_silver.columns)

print("\n--- Silver schema ---")
df_silver.printSchema()

print("\n--- Label distribution (Silver) ---")
label_candidates = [c for c in df_silver.columns if 'label' in c.lower()]
if label_candidates:
    label_col = label_candidates[0]
    df_silver.groupBy(label_col).count().orderBy(F.col("count").desc()).show(20, truncate=False)

print("\n--- Attack category distribution (if available) ---")
if "attack_category" in df_silver.columns:
    df_silver.groupBy("attack_category").count().orderBy(F.col("count").desc()).show()

print("\n--- is_attack distribution (if available) ---")
if "is_attack" in df_silver.columns:
    df_silver.groupBy("is_attack").count().orderBy("is_attack").show()

print("\n--- Null count per column (Silver, first 15 columns) ---")
null_counts_silver = df_silver.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver.columns[:15]
])
null_counts_silver.show(truncate=False)

In [0]:
# === Run Silver -> Gold feature engineering ===
print("Starting feature engineering pipeline...")
print("=" * 60)

df_gold = transform_silver_to_gold(df_silver)

print("=" * 60)
print(f"\n✓ Gold DataFrame ready")
print(f"  Rows: {df_gold.count()}")
print(f"  Columns: {len(df_gold.columns)}")

In [0]:
# === Inspect the Gold DataFrame ===

print("--- New features added in Gold ---")
new_features = [c for c in df_gold.columns if c not in df_silver.columns]
print(f"New features ({len(new_features)}):")
for f in new_features:
    print(f"  - {f}")

print(f"\n--- Gold schema (new features only) ---")
df_gold.select(*new_features).printSchema()

print(f"\n--- Preview of new features ---")
df_gold.select(*new_features).show(5, truncate=False)

print(f"\n--- Label distribution (Gold) ---")
if "label" in df_gold.columns:
    df_gold.groupBy("label").count().orderBy(F.col("count").desc()).show(20, truncate=False)

print(f"\n--- is_attack distribution (Gold) ---")
if "is_attack" in df_gold.columns:
    df_gold.groupBy("is_attack").count().orderBy("is_attack").show()

print(f"\n--- Protocol name distribution (Gold) ---")
if "protocol_name" in df_gold.columns:
    df_gold.groupBy("protocol_name").count().orderBy(F.col("count").desc()).show()

print(f"\n--- Stats on total_packets ---")
if "total_packets" in df_gold.columns:
    df_gold.describe("total_packets", "total_bytes", "bytes_per_packet", "total_flag_count").show()

print(f"\n--- Final Gold column count ---")
print(f"Total columns in Gold: {len(df_gold.columns)}")
print(f"Silver columns: {len(df_silver.columns)}")
print(f"New features added: {len(new_features)}")